# Grand Mock Contest — Contest Python: USACO Bronze Capstone Solutions

Reference solutions for all eight problems. Each is a complete stdin/stdout program; the cells below are display-only mirrors of the solver files in the `assets/` folder.

## Problem 1

**Notice:** build `prefix`, where `prefix[cut]` is the total of the first `cut` boxes. A cut after box `cut` (from `1` to `N - 1`, so both parts keep a box) has left total `prefix[cut]` and right total `total - prefix[cut]`, and the smallest `abs(left - right)` over all cuts is the answer. Negative values need no special care. One pass builds the array and one pass tries every cut, so `100000` boxes take about `2N` steps instead of re-adding both parts for every cut.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
n = int(tokens[0])
prefix = [0]
i = 0
while i < n:
    prefix.append(prefix[i] + int(tokens[1 + i]))
    i = i + 1
total = prefix[n]

best = -1
cut = 1
while cut < n:
    left = prefix[cut]
    right = total - left
    gap = abs(left - right)
    if best == -1 or gap < best:
        best = gap
    cut = cut + 1
print(str(best))


## Problem 2

**Notice:** store the beam's direction as a row step and a column step, starting at `(0, 1)` (moving right). Each time the beam is inside the maze, count one entry and look at the cell: `/` turns `(d_row, d_col)` into `(-d_col, -d_row)`, and `\` turns it into `(d_col, d_row)`, which match the turns in the statement; then step forward. When the beam leaves, the side follows from which bound it crossed. A beam can cross each cell in each of four directions at most once before it leaves, so there are at most `4 * 250000` entries.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
rows = int(tokens[0])
cols = int(tokens[1])
row = int(tokens[2])
grid = []
r = 0
while r < rows:
    grid.append(tokens[3 + r])
    r = r + 1

col = 0
d_row = 0
d_col = 1
entered = 0
while 0 <= row and row < rows and 0 <= col and col < cols:
    entered = entered + 1
    cell = grid[row][col]
    if cell == "/":
        old_row = d_row
        d_row = -d_col
        d_col = -old_row
    elif cell == "\\":
        old_row = d_row
        d_row = d_col
        d_col = old_row
    row = row + d_row
    col = col + d_col

side = "E"
if row < 0:
    side = "N"
elif row >= rows:
    side = "S"
elif col < 0:
    side = "W"
print(str(entered) + " " + side)


## Problem 3

**Notice:** sort the positions. `beacons_placed(spots, gap)` places a beacon at the first spot and then at every spot at least `gap` beyond the last beacon; placing each beacon as early as possible leaves the most room for the rest, so this greedy count is the most beacons that keep every gap at least `gap`. A larger `gap` never allows more beacons, so binary-search the largest `gap` from `1` to `spots[n - 1] - spots[0]` whose count is at least `K`, using `mid = (lo + hi + 1) // 2` so the search always moves. About `30` checks of `N` steps each replace trying every distance up to `10^9`.

In [ ]:
import sys


def beacons_placed(spots, gap):
    placed = 1
    last = spots[0]
    for spot in spots:
        if spot - last >= gap:
            placed = placed + 1
            last = spot
    return placed


data = sys.stdin.read()
tokens = data.split()
n = int(tokens[0])
k = int(tokens[1])
spots = []
i = 0
while i < n:
    spots.append(int(tokens[2 + i]))
    i = i + 1
spots.sort()

lo = 1
hi = spots[n - 1] - spots[0]
while lo < hi:
    mid = (lo + hi + 1) // 2
    if beacons_placed(spots, mid) >= k:
        lo = mid
    else:
        hi = mid - 1
print(str(lo))


## Problem 4

**Notice:** sort the distinct values. For each `lo`, advance `hi` while `values[hi] - values[lo] < D`; then `values[hi]` is the smallest value at least `D` above `values[lo]`, and there is a pair exactly when that difference equals `D`. As `lo` moves right, `values[lo]` grows, so `hi` never needs to move back. Both pointers cross the list once, so after sorting `100000` values take a linear scan instead of checking about five billion pairs.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
n = int(tokens[0])
gap = int(tokens[1])
values = []
i = 0
while i < n:
    values.append(int(tokens[2 + i]))
    i = i + 1
values.sort()

pairs = 0
hi = 0
lo = 0
while lo < n:
    while hi < n and values[hi] - values[lo] < gap:
        hi = hi + 1
    if hi < n and values[hi] - values[lo] == gap:
        pairs = pairs + 1
    lo = lo + 1
print(str(pairs))


## Problem 5

**Notice:** run BFS from `S` with a `deque`, recording `dist` and `ways` for each node. The first time a node `v` is reached from `u`, set `dist[v] = dist[u] + 1` and `ways[v] = ways[u]`. Every later edge from a node `u` with `dist[v] == dist[u] + 1` is another last road of a shortest route, so add `ways[u]`, modulo `1000000007`. BFS takes nodes in order of distance, so every node one road closer is finished before its count is used. The source starts with `0 1`, which covers `S = T`, and an unreached target prints `-1 0`. Each node and road is handled a constant number of times, so `50000` nodes and `100000` roads are one pass.

In [ ]:
from collections import deque
import sys

data = sys.stdin.read()
tokens = data.split()
n = int(tokens[0])
m = int(tokens[1])
source = int(tokens[2])
target = int(tokens[3])
adj = {}
node = 1
while node <= n:
    adj[node] = []
    node = node + 1
i = 0
while i < m:
    u = int(tokens[4 + 2 * i])
    v = int(tokens[5 + 2 * i])
    adj[u].append(v)
    adj[v].append(u)
    i = i + 1

modulus = 1000000007
dist = {source: 0}
ways = {source: 1}
queue = deque()
queue.append(source)
while len(queue) > 0:
    u = queue.popleft()
    for v in adj[u]:
        if v not in dist:
            dist[v] = dist[u] + 1
            ways[v] = ways[u]
            queue.append(v)
        elif dist[v] == dist[u] + 1:
            ways[v] = (ways[v] + ways[u]) % modulus
if target in dist:
    print(str(dist[target]) + " " + str(ways[target]))
else:
    print("-1 0")


## Problem 6

**Notice:** scan the map, and at every unvisited water cell flood-fill its pond with the recursive `fill`, collecting its cells in `found`. The pond is enclosed when none of those cells is in row `0`, row `R - 1`, column `0`, or column `C - 1`; then count the pond and add `len(found)` to the area. The whole pond must be collected before deciding, because a pond may touch the edge far from where the fill started. Each cell is filled once, and the map has at most `400` cells, so the recursion stays shallow and fast.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
rows = int(tokens[0])
cols = int(tokens[1])
grid = []
r = 0
while r < rows:
    grid.append(tokens[2 + r])
    r = r + 1
visited = set()


def fill(cr, cc, found):
    visited.add((cr, cc))
    found.append((cr, cc))
    neighbours = [(cr - 1, cc), (cr + 1, cc), (cr, cc - 1), (cr, cc + 1)]
    for spot in neighbours:
        nr = spot[0]
        nc = spot[1]
        if 0 <= nr and nr < rows and 0 <= nc and nc < cols:
            if grid[nr][nc] == "." and (nr, nc) not in visited:
                fill(nr, nc, found)


ponds = 0
area = 0
r = 0
while r < rows:
    c = 0
    while c < cols:
        if grid[r][c] == "." and (r, c) not in visited:
            found = []
            fill(r, c, found)
            enclosed = True
            for cell in found:
                if cell[0] == 0 or cell[0] == rows - 1 or cell[1] == 0 or cell[1] == cols - 1:
                    enclosed = False
            if enclosed:
                ponds = ponds + 1
                area = area + len(found)
        c = c + 1
    r = r + 1
print(str(ponds) + " " + str(area))


## Problem 7

**Notice:** recursive backtracking that seats each unused person `p` at position `seat` when `p != seat`, marking/unmarking `used`, and counts the complete valid seatings.

In [ ]:
import sys

data = sys.stdin.read()
n = int(data.split()[0])
used = []
build = 0
while build <= n:
    used.append(0)
    build = build + 1

def place(seat):
    if seat > n:
        return 1
    total = 0
    p = 1
    while p <= n:
        if used[p] == 0 and p != seat:
            used[p] = 1
            total = total + place(seat + 1)
            used[p] = 0
        p = p + 1
    return total

print(str(place(1)))


## Problem 8

**Notice:** `visit(node, depth)` returns at once for the `-1` sentinel. Otherwise it adds one to `counts[depth - 1]` and the node's label to `sums[depth - 1]`, appending a new depth the first time that depth is reached, and then visits the left and right children at `depth + 1`. Afterwards, pick the depth with the largest count, replacing the best only on a strictly larger count so ties keep the smallest depth. Each node is visited once, so `100000` nodes take linear time, and the height limit of `500` keeps the recursion under Python's default depth limit.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
n = int(tokens[0])
labels = [0]
left = [-1]
right = [-1]
i = 0
while i < n:
    labels.append(int(tokens[1 + 3 * i]))
    left.append(int(tokens[2 + 3 * i]))
    right.append(int(tokens[3 + 3 * i]))
    i = i + 1

counts = []
sums = []


def visit(node, depth):
    if node == -1:
        return
    if depth > len(counts):
        counts.append(0)
        sums.append(0)
    counts[depth - 1] = counts[depth - 1] + 1
    sums[depth - 1] = sums[depth - 1] + labels[node]
    visit(left[node], depth + 1)
    visit(right[node], depth + 1)


visit(1, 1)
best = 0
level = 1
while level < len(counts):
    if counts[level] > counts[best]:
        best = level
    level = level + 1
print(str(best + 1) + " " + str(sums[best]))
